In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    display_name,
    ensure_dirs,
    normalize_repair,
    normalize_table,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp1_space_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
SELECTED_SERIES = [
    ('naive', 'NR'),
    ('ivmh', 'NR'),
    ('heap', 'WR'),
    ('chain', 'WR'),
    ('par', 'WR'),
]

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.002,
    'probe_ratio': 0.001,
    'scan_reuse_ratio': 0.5,
    'analytical_uniform': True,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'force_rerun': False,
}

WORKLOADS = {
    'RH': 0.80,
    'B': 0.50,
    'WH': 0.20,
}
WORKLOAD_LABELS = {
    'RH': 'Read-Heavy',
    'B': 'Balanced',
    'WH': 'Write-Heavy',
}

RUN_STAMP = current_run_stamp()

RUN_TAG = (
    f"wc{CONFIG['warehouse_count']}_tx{CONFIG['txn_count']}_b{CONFIG['bucket_num']}"
    f"_u{str(CONFIG['update_ratio']).replace('.', 'p')}"
    f"_p{str(CONFIG['probe_ratio']).replace('.', 'p')}"
    f"_re{CONFIG['readable_every']}"
    f"_{RUN_STAMP}"
)

BASE_ARGS = [
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--scan-reuse-ratio', str(CONFIG['scan_reuse_ratio']),
    '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]
if CONFIG['analytical_uniform']:
    BASE_ARGS += ['--analytical-uniform', 'on']

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('STAMP  :', RUN_STAMP)
print('TAG    :', RUN_TAG)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')


In [ ]:
def label_for(table_type, repair_type):
    if table_type in {'naive', 'ivmh'}:
        return display_name(table_type, '')
    return display_name(table_type, repair_type)


def load_space_csv(path):
    df = pd.read_csv(path, keep_default_na=False)
    df['table_type'] = df['table_type'].map(normalize_table)
    df['repair_type'] = df['repair_type'].map(normalize_repair)
    df['current_mib'] = df['current_space'] / (1024 * 1024)
    df['additional_mib'] = (df['total_space'] - df['current_space']) / (1024 * 1024)
    df['total_mib'] = df['total_space'] / (1024 * 1024)
    return df


def run_workload(name, analytical_ratio):
    merged_csv = DATA_DIR / f'sigmod_exp1_space_{name}_{RUN_TAG}.csv'
    if merged_csv.exists() and not CONFIG['force_rerun']:
        print(f'Using cached CSV: {merged_csv.name}')
        return load_space_csv(merged_csv)

    dfs = []
    args = BASE_ARGS + ['--analytical-ratio', str(analytical_ratio)]
    for table_type in TABLE_TYPES:
        print(f'  workload={name} table={table_type}')
        table_csv = DATA_DIR / f'sigmod_exp1_space_{name}_{table_type}_{RUN_TAG}.csv'
        if table_csv.exists() and CONFIG['force_rerun']:
            table_csv.unlink()
        if not table_csv.exists():
            run_checked([
                str(BIN),
                *args,
                '--table-type', table_type,
                '--space-stat', str(table_csv),
            ], ROOT, quiet=True)
        dfs.append(load_space_csv(table_csv))

    merged = pd.concat(dfs, ignore_index=True)
    merged.to_csv(merged_csv, index=False)
    return merged


space_frames = {}
for workload_name, analytical_ratio in WORKLOADS.items():
    space_frames[workload_name] = run_workload(workload_name, analytical_ratio)

space_df = pd.concat(
    [df.assign(workload=workload_name) for workload_name, df in space_frames.items()],
    ignore_index=True,
)
space_df['current_mib'] = space_df['current_space'] / (1024 * 1024)
space_df['additional_mib'] = (space_df['total_space'] - space_df['current_space']) / (1024 * 1024)
space_df['total_mib'] = space_df['total_space'] / (1024 * 1024)

display_cols = [
    'workload', 'table_type', 'repair_type',
    'current_mib', 'additional_mib', 'total_mib'
]
display(space_df[display_cols].round(2))


In [ ]:
COLOR_MAP = {
    'Current': TOL['grey'],
    'Historical Support': TOL['blue'],
}


def selected_rows(df):
    rows = []
    for table_type, repair_type in SELECTED_SERIES:
        row = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)]
        if row.empty:
            raise ValueError(f'Missing row for {(table_type, repair_type)}')
        rows.append(row.iloc[0])
    return pd.DataFrame(rows)


workload_order = ['RH', 'B', 'WH']
stack_order = ['Current', 'Historical Support']
global_y_max = max(float(selected_rows(space_frames[w])['total_mib'].max()) for w in workload_order) * 1.12

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
for ax, workload_name in zip(axes, workload_order):
    plot_df = selected_rows(space_frames[workload_name]).copy()
    plot_df['label'] = [label_for(t, r) for t, r in zip(plot_df['table_type'], plot_df['repair_type'])]
    x = np.arange(len(plot_df))
    bottoms = np.zeros(len(plot_df))
    values = {
        'Current': plot_df['current_mib'].to_numpy(),
        'Historical Support': plot_df['additional_mib'].to_numpy(),
    }
    for key in stack_order:
        ax.bar(x, values[key], bottom=bottoms, width=0.62, color=COLOR_MAP[key], edgecolor='black', linewidth=0.5, label=key)
        bottoms += values[key]
    for xi, total in zip(x, plot_df['total_mib'].to_numpy()):
        ax.text(xi, total + 0.02 * global_y_max, f'{total:.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(WORKLOAD_LABELS[workload_name])
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['label'], rotation=0)
    ax.set_ylim(0, global_y_max)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

axes[0].set_ylabel('Derived-State Memory (MiB)')
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=COLOR_MAP[key], edgecolor='black', linewidth=0.5) for key in stack_order]
fig.legend(handles, stack_order, ncol=2, loc='upper center', frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=[0, 0, 1, 0.93])

pdf_path = FIGS_DIR / f'sigmod_exp1_space_breakdown_{RUN_TAG}.pdf'
png_path = FIGS_DIR / f'sigmod_exp1_space_breakdown_{RUN_TAG}.png'
latest_pdf = FIGS_DIR / 'exp1-space-overview.pdf'
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(latest_pdf, bbox_inches='tight')
fig.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print('saved:', pdf_path)
print('saved:', latest_pdf)
print('saved:', png_path)


In [ ]:
LATEST_PDFS = {
    'RH': FIGS_DIR / 'exp1-space-read-heavy.pdf',
    'B': FIGS_DIR / 'exp1-space-balanced.pdf',
    'WH': FIGS_DIR / 'exp1-space-write-heavy.pdf',
}

for workload_name in workload_order:
    plot_df = selected_rows(space_frames[workload_name]).copy()
    plot_df['label'] = [label_for(t, r) for t, r in zip(plot_df['table_type'], plot_df['repair_type'])]

    fig, ax = plt.subplots(1, 1, figsize=(6.4, 4.2))
    x = np.arange(len(plot_df))
    bottoms = np.zeros(len(plot_df))
    values = {
        'Current': plot_df['current_mib'].to_numpy(),
        'Historical Support': plot_df['additional_mib'].to_numpy(),
    }
    for key in stack_order:
        ax.bar(x, values[key], bottom=bottoms, width=0.62, color=COLOR_MAP[key], edgecolor='black', linewidth=0.5, label=key)
        bottoms += values[key]

    for xi, total in zip(x, plot_df['total_mib'].to_numpy()):
        ax.text(xi, total + 0.02 * global_y_max, f'{total:.1f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Derived-State Memory (MiB)')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['label'], rotation=0)
    ax.set_ylim(0, global_y_max)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.legend(ncol=2, loc='upper right', frameon=True)
    fig.tight_layout()

    pdf_path = FIGS_DIR / f'sigmod_exp1_space_breakdown_{workload_name.lower()}_{RUN_TAG}.pdf'
    png_path = FIGS_DIR / f'sigmod_exp1_space_breakdown_{workload_name.lower()}_{RUN_TAG}.png'
    latest_pdf = LATEST_PDFS[workload_name]
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(latest_pdf, bbox_inches='tight')
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('saved:', pdf_path)
    print('saved:', latest_pdf)
    print('saved:', png_path)


In [ ]:
ivmh_only = (
    space_df[(space_df['table_type'] == 'ivmh') & (space_df['repair_type'] == 'NR')]
    .copy()
    .sort_values('workload')
)
ivmh_only['cached_total_mib'] = ivmh_only['total_mib']
ivmh_only['uncached_current_only_mib'] = ivmh_only['current_mib']
ivmh_only['extra_from_caching_mib'] = ivmh_only['additional_mib']
ivmh_only['space_saved_without_cache_pct'] = (
    100.0 * ivmh_only['extra_from_caching_mib'] / ivmh_only['cached_total_mib']
)

summary_cols = [
    'workload',
    'cached_total_mib',
    'uncached_current_only_mib',
    'extra_from_caching_mib',
    'space_saved_without_cache_pct',
]
display(ivmh_only[summary_cols].round(2))

rh_row = ivmh_only[ivmh_only['workload'] == 'RH'].iloc[0]
print(
    'RH text candidate:',
    f"Without caching historical snapshots, IVMH would use about {rh_row['uncached_current_only_mib']:.1f} MiB instead of {rh_row['cached_total_mib']:.1f} MiB, "
    f"saving {rh_row['extra_from_caching_mib']:.1f} MiB ({rh_row['space_saved_without_cache_pct']:.1f}%)."
)
